# 03 — Sobreajuste, regularización y validación

## Motivación

En el ejercicio de la sesión pasada viste un modelo que **se queda corto**: una recta
intentando describir una parábola. La solución parece obvia — usar un modelo más
flexible. Pero la flexibilidad tiene un costo: un modelo suficientemente flexible
puede ajustar *perfectamente* los datos de entrenamiento... **memorizando el ruido**.
Ese modelo obtiene pérdida cero en train y generaliza mal con datos nuevos.

Este fenómeno se llama **sobreajuste** (*overfitting*) y es el problema central del
machine learning. La sesión de hoy construye las tres herramientas para manejarlo:

1. **Diagnóstico** — comparar el error de train contra el de test
2. **Tratamiento** — la regularización
3. **Protocolo** — la validación cruzada para elegir hiperparámetros con honestidad

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("../../datos")

rng = np.random.default_rng(seed=42)

## El experimento: ajustar polinomios a una ley conocida

Otra vez el experimento controlado. La ley verdadera será

$$y = \sin(2\pi x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 0.3^2)$$

con apenas **25 puntos** de entrenamiento. Ajustaremos polinomios de grado creciente:

$$f(x) = w_0 + w_1 x + w_2 x^2 + \cdots + w_g x^g$$

Nota: esto **sigue siendo regresión lineal** — el modelo es lineal *en los parámetros*
$w_j$. Solo cambiamos las características: en lugar de $x$, usamos $(x, x^2, \ldots, x^g)$.
En scikit-learn eso es `PolynomialFeatures` seguido de `LinearRegression`,
encadenados en un `Pipeline`.

In [ ]:
def true_function(x):
    """Ley que genera los datos (en la práctica nunca la conocemos)."""
    return np.sin(2 * np.pi * x)


noise_std = 0.3
n_train, n_test = 25, 300

x_train = rng.uniform(0, 1, n_train)
y_train = true_function(x_train) + rng.normal(0, noise_std, n_train)

x_test = rng.uniform(0, 1, n_test)
y_test = true_function(x_test) + rng.normal(0, noise_std, n_test)

x_grid = np.linspace(0, 1, 300)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_train, y=y_train, mode="markers", name="train (25 puntos)",
    marker=dict(size=8, opacity=0.8),
))
fig.add_trace(go.Scatter(
    x=x_grid, y=true_function(x_grid), mode="lines", name="ley verdadera",
    line=dict(width=2, color="#00CC96"),
))
fig.update_layout(
    title="Pocos datos, ruido real: el escenario típico",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
)
fig.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures


def fit_polynomial(degree, x, y):
    """Ajusta un polinomio del grado dado vía regresión lineal sobre potencias de x."""
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x.reshape(-1, 1), y)
    return model


degrees_to_show = [1, 3, 15]

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"grado {d}" for d in degrees_to_show])

for col, degree in enumerate(degrees_to_show, start=1):
    model = fit_polynomial(degree, x_train, y_train)
    fig.add_trace(
        go.Scatter(x=x_train, y=y_train, mode="markers",
                    marker=dict(size=6, opacity=0.7, color="#636EFA"), showlegend=False),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=true_function(x_grid), mode="lines",
                    line=dict(width=1.5, color="#00CC96"), opacity=0.7, showlegend=False),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=model.predict(x_grid.reshape(-1, 1)), mode="lines",
                    line=dict(width=2, color="#EF553B"), showlegend=False),
        row=1, col=col,
    )
    fig.update_yaxes(range=[-1.8, 1.8], row=1, col=col)
    fig.update_xaxes(title_text="x", row=1, col=col)

fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_layout(
    title="Subajuste (grado 1) — ajuste razonable (3) — sobreajuste (15)",
    template="plotly_white",
    height=420,
)
fig.show()

El panel resume el fenómeno completo:

- **Grado 1** (recta): demasiado rígido — ni siquiera describe el train. *Subajuste.*
- **Grado 3**: captura la forma de la ley verdadera y suaviza el ruido.
- **Grado 15**: pasa casi exactamente por cada punto de train — **incluido el ruido** —
  y entre puntos oscila con amplitud creciente. *Sobreajuste.*

El detalle crucial: el modelo de grado 15 tiene el **menor** error de entrenamiento de
los tres. Si solo miráramos train, lo elegiríamos. Cuantifiquemos con la curva de
validación: error de train y de test como función del grado.

In [ ]:
from sklearn.metrics import root_mean_squared_error

degrees = range(0, 16)
rmse_train, rmse_test = [], []

for degree in degrees:
    model = fit_polynomial(degree, x_train, y_train)
    pred_train = model.predict(x_train.reshape(-1, 1))
    pred_test = model.predict(x_test.reshape(-1, 1))
    rmse_train.append(root_mean_squared_error(y_train, pred_train))
    rmse_test.append(root_mean_squared_error(y_test, pred_test))

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(degrees), y=rmse_train, mode="lines+markers", name="train"))
fig.add_trace(go.Scatter(x=list(degrees), y=rmse_test, mode="lines+markers", name="test"))
fig.add_hline(
    y=noise_std, line_dash="dash", line_color="#00CC96",
    annotation_text="ruido irreducible (σ)", annotation_position="bottom right",
)
fig.update_layout(
    title="La curva de validación: train baja siempre, test tiene un mínimo",
    xaxis_title="grado del polinomio (complejidad del modelo)",
    yaxis_title="RMSE",
    yaxis_type="log",
    template="plotly_white",
)
fig.show()

## Teoría: la descomposición sesgo–varianza

La curva anterior obedece a una ley general, no a una particularidad de este dataset.
Para el error cuadrático esperado de un modelo $\hat{f}$ entrenado sobre datasets
aleatorios, en un punto $x$:

$$\mathbb{E}\left[ (y - \hat{f}(x))^2 \right]
= \underbrace{\left( f(x) - \mathbb{E}[\hat{f}(x)] \right)^2}_{\text{sesgo}^2}
+ \underbrace{\mathbb{E}\left[ \left( \hat{f}(x) - \mathbb{E}[\hat{f}(x)] \right)^2 \right]}_{\text{varianza}}
+ \underbrace{\sigma^2}_{\text{ruido}}$$

(la derivación completa está en el **Apéndice A**). La esperanza $\mathbb{E}[\hat f(x)]$ es
sobre **datasets de entrenamiento distintos**, todos generados por el mismo proceso: es
la formalización de "qué pasaría si entrenáramos el modelo una y otra vez con muestras
nuevas". Cada término es una cantidad calculable, no una etiqueta cualitativa:

- **Sesgo** — $f(x) - \mathbb{E}[\hat{f}(x)]$: se fija un punto $x$, se entrena el
  modelo sobre muchos datasets distintos del mismo proceso, se promedian las
  predicciones en $x$, y se compara ese promedio contra el valor verdadero $f(x)$. La
  recta (grado 1) tiene sesgo alto: su promedio sobre infinitos datasets sigue siendo
  una recta, y ninguna recta coincide con $\sin(2\pi x)$.
- **Varianza** — $\mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]$: la
  dispersión de esas mismas predicciones **alrededor de su propio promedio** — no
  contra la verdad, contra sí mismas. El polinomio de grado 15 tiene varianza alta:
  entrenado sobre 25 puntos distintos del mismo proceso, da 25 curvas muy distintas
  entre sí, porque cada una se ajusta al ruido específico de su muestra.
- **Ruido irreducible** $\sigma^2$: la varianza de $\varepsilon$ en $y = f(x) + \varepsilon$.
  No depende del modelo elegido — es la aleatoriedad del proceso que genera los datos.
  Ni siquiera usar $f$ exacta como predictor evita este término: los $y$ nuevos siguen
  trayendo ruido.

Aumentar la complejidad reduce el sesgo y aumenta la varianza: por eso el error
de test tiene forma de U. Elegir un modelo es elegir un punto en ese equilibrio.

### Midiendo sesgo y varianza con números

La definición anterior dice "entrenar sobre muchos datasets distintos"; esta sección
lo hace de forma explícita. El proceso generador $y = \sin(2\pi x) + \varepsilon$ se
conoce por construcción, así que se pueden generar 200 datasets de entrenamiento
independientes, ajustar cada grado sobre cada uno, y calcular sesgo y varianza tal
como se definieron, en vez de solo describirlos.

In [ ]:
n_resamples = 200
degrees_bv = [1, 15]

predictions = {d: np.empty((n_resamples, len(x_grid))) for d in degrees_bv}
for i in range(n_resamples):
    x_i = rng.uniform(0, 1, n_train)
    y_i = true_function(x_i) + rng.normal(0, noise_std, n_train)
    for d in degrees_bv:
        model = fit_polynomial(d, x_i, y_i)
        predictions[d][i] = model.predict(x_grid.reshape(-1, 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=[f"grado {d}" for d in degrees_bv])
for col, d in enumerate(degrees_bv, start=1):
    for i in range(40):  # se muestran 40 de los 200 ajustes, el resto satura la figura
        fig.add_trace(
            go.Scatter(x=x_grid, y=predictions[d][i], mode="lines",
                        line=dict(width=1, color="#636EFA"), opacity=0.15, showlegend=False),
            row=1, col=col,
        )
    fig.add_trace(
        go.Scatter(x=x_grid, y=predictions[d].mean(axis=0), mode="lines",
                    name="promedio de los ajustes", line=dict(width=2.5, color="#EF553B"),
                    showlegend=(col == 1)),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=true_function(x_grid), mode="lines", name="ley verdadera",
                    line=dict(width=2, color="#00CC96", dash="dash"), showlegend=(col == 1)),
        row=1, col=col,
    )
    fig.update_yaxes(range=[-2.2, 2.2], row=1, col=col)
    fig.update_xaxes(title_text="x", row=1, col=col)

fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_layout(
    title="200 ajustes sobre datasets distintos del mismo proceso (se muestran 40)",
    template="plotly_white",
    height=430,
)
fig.show()

En grado 1 (izquierda), las 40 rectas casi se superponen — poca varianza — pero su
promedio (rojo) queda lejos de la ley verdadera (verde): sesgo alto. En grado 15
(derecha) ocurre lo opuesto: las curvas individuales son muy distintas entre sí —
varianza alta — aunque su promedio se acerca bastante a la ley verdadera: sesgo bajo.
Verifiquemos con los números exactos de la definición, y que en efecto suman el error
observado.

In [ ]:
# Sesgo² y varianza, calculados exactamente como se definieron arriba
print(f"{'grado':>6}  {'sesgo²':>9}  {'varianza':>9}  {'ruido':>7}  {'suma':>9}   {'error simulado':>15}")
for d in degrees_bv:
    mean_pred = predictions[d].mean(axis=0)
    bias2 = np.mean((true_function(x_grid) - mean_pred) ** 2)
    variance = np.mean(predictions[d].var(axis=0))
    noise = noise_std**2
    suma = bias2 + variance + noise

    # error real: cada uno de los 200 ajustes contra una observación NUEVA con ruido fresco
    y_nuevo = true_function(x_grid) + rng.normal(0, noise_std, size=(n_resamples, len(x_grid)))
    error_simulado = np.mean((y_nuevo - predictions[d]) ** 2)

    print(f"{d:>6}  {bias2:>9.4f}  {variance:>9.4f}  {noise:>7.4f}  {suma:>9.4f}   {error_simulado:>15.4f}")

Las columnas *suma* y *error simulado* deben coincidir — son la misma cantidad
calculada por dos caminos independientes: sumando sesgo²+varianza+ruido según la
fórmula, o simulando directamente muchas observaciones nuevas y midiendo el error
sobre ellas. Que coincidan es la verificación numérica de la descomposición del
Apéndice A. El grado 1 tiene sesgo² dominante y varianza casi nula; el grado 15, al
revés: sesgo² pequeño y varianza que domina por completo la suma — el sobreajuste
medido en números.

## Regularización: penalizar la complejidad

¿Y si en lugar de restringir el *grado* restringimos los *pesos*? El sobreajuste de
grado 15 se refleja en sus coeficientes: para oscilar entre los puntos de ruido necesita
pesos enormes que se cancelan entre sí. La **regularización** penaliza justo eso —
agrega a la pérdida un costo por el tamaño de los pesos:

$$L_{\text{ridge}}(\mathbf{w}) = \lVert X\mathbf{w} - \mathbf{y} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_2^2
\qquad \text{(Ridge, penalización L2)}$$

$$L_{\text{lasso}}(\mathbf{w}) = \lVert X\mathbf{w} - \mathbf{y} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_1
\qquad \text{(Lasso, penalización L1)}$$

El **hiperparámetro** $\alpha \geq 0$ controla la intensidad de la penalización:
$\alpha = 0$ recupera mínimos cuadrados; a medida que $\alpha$ crece, los pesos se
reducen hacia cero. Este efecto se llama ***shrinkage*** (contracción) en la
literatura de aprendizaje estadístico: Ridge y Lasso aceptan un aumento pequeño de
sesgo a cambio de una reducción mayor de varianza (Hastie, Tibshirani y Friedman,
*The Elements of Statistical Learning*, cap. 3).

**Requisito práctico**: las penalizaciones comparan pesos entre sí, así que las
características deben estar en una escala común. De aquí en adelante, `StandardScaler`
(restar media, dividir entre desviación estándar) es parte estándar del pipeline.

### Por qué Lasso fija coeficientes en cero y Ridge no

Minimizar $L_{\text{ridge}}$ o $L_{\text{lasso}}$ equivale a minimizar el error
$\lVert X\mathbf{w} - \mathbf{y}\rVert^2$ sujeto a una restricción sobre el tamaño de
$\mathbf{w}$: $\lVert\mathbf{w}\rVert_2 \le t$ para Ridge, $\lVert\mathbf{w}\rVert_1 \le t$
para Lasso (a cada $\alpha$ le corresponde un $t$). En el espacio de dos pesos
$(w_1, w_2)$:

- La región $\lVert\mathbf{w}\rVert_2 \le t$ es un **círculo**.
- La región $\lVert\mathbf{w}\rVert_1 \le t$ es un **rombo**, con vértices sobre los ejes.

La solución regularizada es el punto de esa región donde toca por primera vez una
curva de nivel (una elipse) del error sin restringir. El círculo no tiene esquinas:
casi nunca esas elipses lo tocan exactamente sobre un eje, así que **ambos** pesos
quedan distintos de cero. El rombo sí tiene esquinas sobre los ejes, y las elipses
tienden a tocarlo justo ahí — en esa esquina, uno de los pesos **es** cero. Esa
asimetría geométrica, no una regla arbitraria, es lo que hace que Lasso seleccione
características y Ridge no.

Para verlo con números en vez de solo en la geometría: dos características casi
duplicadas — la superficie de una casa en $\text{ft}^2$ y en $\text{m}^2$,
prácticamente la misma información en dos escalas — prediciendo una respuesta que
depende de esa superficie más ruido.

In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

# Generador propio (no el `rng` compartido): esta demo debe dar el mismo resultado
# sin importar qué otras celdas se hayan ejecutado antes.
rng_house = np.random.default_rng(0)

sqft = rng_house.normal(1500, 300, 200)
sqm = sqft * 0.0929 + rng_house.normal(0, 0.5, 200)  # casi duplicado, con ruido de medición
y_house = 50_000 + 120 * sqft + rng_house.normal(0, 15_000, 200)

X_house = StandardScaler().fit_transform(np.column_stack([sqft, sqm]))
y_house_s = StandardScaler().fit_transform(y_house.reshape(-1, 1)).ravel()

ols_w = LinearRegression().fit(X_house, y_house_s).coef_
print(f"OLS sin regularizar:  w_sqft = {ols_w[0]:.3f}   w_sqm = {ols_w[1]:.3f}  (inestable: signos opuestos)")

alphas_path = np.logspace(-3, 1, 30)
ridge_path = np.array([Ridge(alpha=a).fit(X_house, y_house_s).coef_ for a in alphas_path])
lasso_path = np.array([Lasso(alpha=a, max_iter=20000).fit(X_house, y_house_s).coef_ for a in alphas_path])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Ridge", "Lasso"))
for col, path in enumerate([ridge_path, lasso_path], start=1):
    fig.add_trace(
        go.Scatter(x=path[:, 0], y=path[:, 1], mode="lines+markers",
                    marker=dict(size=4), line=dict(width=2), showlegend=False),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=[ols_w[0]], y=[ols_w[1]], mode="markers", name="OLS (α=0)",
                    marker=dict(size=11, symbol="x", color="#EF553B"), showlegend=(col == 1)),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=[0], y=[0], mode="markers", name="origen (α→∞)",
                    marker=dict(size=9, symbol="circle-open", color="black"), showlegend=(col == 1)),
        row=1, col=col,
    )
    fig.update_xaxes(title_text="w_sqft", row=1, col=col)

fig.update_yaxes(title_text="w_sqm", row=1, col=1)
fig.update_layout(
    title="Trayectoria de los pesos al crecer α (características casi duplicadas)",
    template="plotly_white",
    height=430,
)
fig.show()

Sin regularizar, mínimos cuadrados da pesos inestables — uno positivo y grande, el
otro negativo, que se cancelan casi por completo. Es la firma clásica de la
multicolinealidad: el modelo encuentra una combinación frágil que ajusta el ruido de
esta muestra en particular, no la relación real.

En la gráfica, según $\alpha$ crece desde ese punto (×) hacia el origen:

- **Ridge** llega a un punto donde **ambos** pesos son positivos y de tamaño
  parecido: reparte el peso entre las dos versiones de la misma variable, sin fijar
  ninguna en cero.
- **Lasso** fija uno de los dos pesos en cero casi de inmediato y se queda solo con
  el otro, hasta que $\alpha$ es lo bastante grande para fijar ambos en cero. Cuál de
  los dos llega a cero primero depende de detalles finos del ruido de esta muestra —
  con otra muestra del mismo proceso pudo haber sido el otro. Esa es la inestabilidad
  de Lasso frente a características muy correlacionadas.

### ¿Cuándo usar cuál?

- **Ridge**: cuando se espera que *muchas* características aporten algo, aunque sea
  poco cada una, y hay características correlacionadas entre sí — reparte el peso
  entre ellas en vez de elegir una arbitrariamente. Da el modelo más **estable** frente
  a pequeños cambios en los datos de entrenamiento.
- **Lasso**: cuando se sospecha que *pocas* características importan y el resto es
  ruido — produce un modelo más simple e interpretable al eliminarlas. El costo es la
  inestabilidad que se acaba de ver con características correlacionadas.
- **Elastic Net** (`sklearn.linear_model.ElasticNet`, no se usa en este notebook):
  combina ambas penalizaciones. Es la opción práctica habitual cuando hay muchas
  características correlacionadas y aun así se quiere selección — reparte el peso
  entre las correlacionadas (como Ridge) mientras fija en cero las irrelevantes (como
  Lasso).

En la práctica la elección no se hace por intuición sino por validación cruzada,
comparando el error de cada opción — el tema de la siguiente sección. Volvamos primero
al problema original: el efecto de regularizar sobre el polinomio de grado 15.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

alphas_to_show = [0, 1e-4, 1]
degree = 15

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"grado 15, α = {a}" for a in alphas_to_show])

for col, alpha in enumerate(alphas_to_show, start=1):
    model = make_pipeline(
        PolynomialFeatures(degree),
        StandardScaler(),
        Ridge(alpha=alpha) if alpha > 0 else LinearRegression(),
    )
    model.fit(x_train.reshape(-1, 1), y_train)
    fig.add_trace(
        go.Scatter(x=x_train, y=y_train, mode="markers",
                    marker=dict(size=6, opacity=0.7, color="#636EFA"), showlegend=False),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=true_function(x_grid), mode="lines",
                    line=dict(width=1.5, color="#00CC96"), opacity=0.7, showlegend=False),
        row=1, col=col,
    )
    fig.add_trace(
        go.Scatter(x=x_grid, y=model.predict(x_grid.reshape(-1, 1)), mode="lines",
                    line=dict(width=2, color="#EF553B"), showlegend=False),
        row=1, col=col,
    )
    fig.update_yaxes(range=[-1.8, 1.8], row=1, col=col)
    fig.update_xaxes(title_text="x", row=1, col=col)

fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_layout(
    title="El mismo polinomio de grado 15: mayor α reduce la varianza, aumenta el sesgo",
    template="plotly_white",
    height=420,
)
fig.show()

## ¿Cómo elegir α? Validación cruzada

Tenemos que elegir $\alpha$ (y el grado, y en general los **hiperparámetros**: los que
el ajuste no aprende). La tentación es probar valores y quedarnos con el que dé mejor
error **de test**, pero eso invalida el test como medida de generalización: si lo usamos
para decidir, habríamos ajustado los hiperparámetros a él.

**Principio central: el test se toca una sola vez, al final de todo.**

La herramienta correcta es la **validación cruzada** (*k-fold CV*) dentro del train:

1. Divide el train en $k$ bloques (típicamente $k = 5$).
2. Entrena con $k-1$ bloques y valida con el restante; rota $k$ veces.
3. Promedia los $k$ errores de validación.

Cada punto se usa para validar exactamente una vez, y el promedio es mucho más estable
que un split único — especialmente con pocos datos. `GridSearchCV` automatiza el
recorrido sobre una rejilla de hiperparámetros con este protocolo.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    PolynomialFeatures(),
    StandardScaler(),
    Ridge(),
)

param_grid = {
    "polynomialfeatures__degree": range(1, 16),
    "ridge__alpha": np.logspace(-6, 2, 9),
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
search.fit(x_train.reshape(-1, 1), y_train)

best_rmse_cv = -search.best_score_
print(f"mejores hiperparámetros: {search.best_params_}")
print(f"RMSE de validación cruzada: {best_rmse_cv:.3f}")

# Único uso del test, después de fijar los hiperparámetros por validación cruzada:
rmse_final = root_mean_squared_error(y_test, search.predict(x_test.reshape(-1, 1)))
print(f"RMSE final en test:         {rmse_final:.3f}   (σ del ruido = {noise_std})")

## De vuelta al mundo real: California Housing regularizado

Cerramos aplicando el protocolo completo — pipeline con escalado, búsqueda de $\alpha$
por validación cruzada, evaluación final en test — al dataset de la sesión pasada.
De paso, con las características **escaladas**, los coeficientes por fin son
comparables entre sí.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(data_home=DATA_DIR, as_frame=True)
X = housing.frame.drop(columns="MedHouseVal")
y = housing.frame["MedHouseVal"]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = {}
for name, estimator in [("ridge", Ridge()), ("lasso", Lasso(max_iter=5000))]:
    pipe = make_pipeline(StandardScaler(), estimator)
    search_h = GridSearchCV(
        pipe,
        {f"{name}__alpha": np.logspace(-4, 2, 13)},
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    search_h.fit(X_train_h, y_train_h)
    rmse = root_mean_squared_error(y_test_h, search_h.predict(X_test_h))
    results[name] = search_h
    alpha_best = search_h.best_params_[f"{name}__alpha"]
    print(f"{name}:  mejor α = {alpha_best:.4f}   RMSE test = {rmse:.3f} [$100k]")

In [ ]:
coef_table = pd.DataFrame(
    {
        name: pd.Series(
            search.best_estimator_[-1].coef_, index=X.columns
        )
        for name, search in results.items()
    }
).sort_values("ridge")

fig = go.Figure()
fig.add_trace(go.Bar(y=coef_table.index, x=coef_table["ridge"], name="ridge", orientation="h"))
fig.add_trace(go.Bar(y=coef_table.index, x=coef_table["lasso"], name="lasso", orientation="h"))
fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(
    title="Ridge encoge todos los coeficientes; Lasso fija algunos exactamente en cero",
    xaxis_title="coeficiente (features escaladas, comparables entre sí)",
    barmode="group",
    template="plotly_white",
)
fig.show()

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **Curva de validación con CV.** Repite la curva train/test del inicio, pero usando
   `cross_val_score` (5 folds) sobre el train en lugar del conjunto de test. ¿El grado
   óptimo coincide con el que encontró `GridSearchCV`?

2. **Lasso como detector de características irrelevantes.** Genera un dataset sintético
   con 10 características donde solo 3 afectan a $y$ (las otras 7 son ruido puro).
   Ajusta Lasso con varios valores de $\alpha$ y grafica los coeficientes.
   ¿Recupera cuáles características importan?

3. **Reto — el tamaño del dataset importa.** Repite el experimento de los polinomios con
   $n_{\text{train}} = 500$ en lugar de 25. ¿Sigue sobreajustando el grado 15?
   Conecta tu respuesta con el término de varianza de la descomposición.

## Apéndice A — Derivación de la descomposición sesgo–varianza

Sea $y = f(x) + \varepsilon$ con $\mathbb{E}[\varepsilon] = 0$,
$\operatorname{Var}(\varepsilon) = \sigma^2$, y sea $\hat{f}$ el modelo entrenado sobre
un dataset aleatorio $\mathcal{D}$ (la esperanza $\mathbb{E}$ es sobre datasets y ruido;
escribimos $\bar{f}(x) \equiv \mathbb{E}[\hat{f}(x)]$).

Sumamos y restamos $\bar{f}(x)$ dentro del error:

$$\mathbb{E}\big[ (y - \hat{f})^2 \big]
= \mathbb{E}\big[ (f + \varepsilon - \bar{f} + \bar{f} - \hat{f})^2 \big]$$

Expandimos el cuadrado del trinomio $(A + B + C)^2$ con $A = f - \bar{f}$ (constante),
$B = \varepsilon$, $C = \bar{f} - \hat{f}$:

$$= \underbrace{(f - \bar{f})^2}_{A^2}
+ \underbrace{\mathbb{E}[\varepsilon^2]}_{B^2}
+ \underbrace{\mathbb{E}\big[ (\bar{f} - \hat{f})^2 \big]}_{C^2}
+ 2\,\mathbb{E}[AB] + 2\,\mathbb{E}[AC] + 2\,\mathbb{E}[BC]$$

Los tres términos cruzados se anulan:

- $\mathbb{E}[AB] = (f - \bar{f})\,\mathbb{E}[\varepsilon] = 0$ — el ruido tiene media cero.
- $\mathbb{E}[AC] = (f - \bar{f})\,\mathbb{E}[\bar{f} - \hat{f}] = (f - \bar{f})(\bar{f} - \bar{f}) = 0$
  — por definición de $\bar{f}$.
- $\mathbb{E}[BC] = 0$ — el ruido del punto de evaluación es independiente del dataset
  de entrenamiento.

Queda exactamente

$$\mathbb{E}\big[ (y - \hat{f})^2 \big]
= \underbrace{(f - \bar{f})^2}_{\text{sesgo}^2}
+ \underbrace{\sigma^2}_{\text{ruido}}
+ \underbrace{\mathbb{E}\big[ (\hat{f} - \bar{f})^2 \big]}_{\text{varianza}} \qquad \blacksquare$$